## Reading data for Spain

In [1]:
import pandas as pd

xls = pd.read_excel('../Extacted data/Pharm stats /Spain_atc3_2021_2025.xlsx', sheet_name=None)
print(xls.keys())

dict_keys(['ENVASES (miles)', 'PVP (miles EUR)'])


In [2]:
# have null value in column 'CHEMICAL_SUBSTANCE_BNF_DESCR'
df1 = xls['ENVASES (miles)']
df1.head()

,code,description_es,description_en,jan-2021,feb-2021,mar-2021,apr-2021,may-2021,jun-2021,jul-2021,...,mar-2025,apr-2025,may-2025,jun-2025,jul-2025,aug-2025,sep-2025,oct-2025,nov-2025,dec-2025
0,A01A,PREPARADOS ESTOMATOLÓGICOS,Stomatological preparations,13.17,14.93,17.25,15.26,16.31,17.03,16.63,...,19.43,18.89,19.42,16.84,11.49,6.66,5.33,3.32,1.56,0.79
1,A02B,AGENTES CONTRA LA ÚLCERA PÉPTICA Y EL REFLUJO ...,Drugs for peptic ulcer and gastro-oesophageal ...,5801.42,5557.14,6452.69,5904.73,5982.98,6064.39,6099.80,...,6065.69,5977.74,6059.48,5772.65,6149.48,5595.66,5980.83,6145.03,5651.30,6153.47
2,A03A,AGENTES CONTRA PADECIMIENTOS FUNCIONALES DEL E...,Drugs for functional gastrointestinal disorders,81.28,84.08,95.35,88.46,90.61,92.39,92.03,...,99.71,96.55,100.42,95.28,101.82,90.75,98.31,104.59,96.87,98.87
3,A03B,"BELLADONA Y DERIVADOS, MONOFÁRMACOS","Belladonna and derivatives, plain",57.66,58.65,64.81,61.24,62.78,63.71,65.82,...,81.30,77.08,80.86,77.28,85.33,82.01,83.62,89.40,83.18,84.00
4,A03F,PROPULSIVOS,Propulsives,447.53,436.87,501.41,464.51,467.64,498.33,506.23,...,555.97,530.22,538.84,516.49,554.29,503.15,535.35,584.23,544.21,574.69


## Fill in nulls in CHEMICAL_SUBSTANCE_BNF_DESCR column from BNF_CHEMICAL_SUBSTANCE

In [3]:
# filter only antidepressants
df_antidepressants = df1[df1["description_en"].isin(["Antidepressants", "Anxiolytics"])]

In [4]:
df_antidepressants

,code,description_es,description_en,jan-2021,feb-2021,mar-2021,apr-2021,may-2021,jun-2021,jul-2021,...,mar-2025,apr-2025,may-2025,jun-2025,jul-2025,aug-2025,sep-2025,oct-2025,nov-2025,dec-2025
132,N05B,ANSIOLÍTICOS,Anxiolytics,4802.61,4619.32,5296.63,4896.67,4917.09,4952.42,4955.85,...,4589.21,4490.54,4568.76,4325.21,4564.88,4179.57,4434.95,4564.21,4157.13,4449.77
134,N06A,ANTIDEPRESIVOS,Antidepressants,3892.41,3720.43,4335.07,4022.20,4082.91,4158.32,4211.59,...,5018.19,4960.36,5068.57,4840.89,5197.63,4723.19,5056.89,5219.98,4822.68,5213.79


In [5]:
df_long = df_antidepressants.melt(
    id_vars=["code", "description_es", "description_en"],
    var_name="date",
    value_name="items_1000"
)
df_long.head()

,code,description_es,description_en,date,items_1000
0,N05B,ANSIOLÍTICOS,Anxiolytics,jan-2021,4802.61
1,N06A,ANTIDEPRESIVOS,Antidepressants,jan-2021,3892.41
2,N05B,ANSIOLÍTICOS,Anxiolytics,feb-2021,4619.32
3,N06A,ANTIDEPRESIVOS,Antidepressants,feb-2021,3720.43
4,N05B,ANSIOLÍTICOS,Anxiolytics,mar-2021,5296.63


## Dropping columns 

In [6]:
df = df_long.drop(columns=['description_es','code'])

In [7]:
df = df.rename(columns={'description_en': 'group',})
df = df[["date", "group", "items_1000"]]
df.head(2)

,date,group,items_1000
0,jan-2021,Anxiolytics,4802.61
1,jan-2021,Antidepressants,3892.41


In [8]:
df['date'] = pd.to_datetime(df['date'], format='%b-%Y').dt.strftime('%Y-%m')

In [9]:
df['country']= 'Spain' 
df['items'] = df['items_1000']*1000
df.head()

,date,group,items_1000,country,items
0,2021-01,Anxiolytics,4802.61,Spain,4802610.0
1,2021-01,Antidepressants,3892.41,Spain,3892410.0
2,2021-02,Anxiolytics,4619.32,Spain,4619320.0
3,2021-02,Antidepressants,3720.43,Spain,3720430.0
4,2021-03,Anxiolytics,5296.63,Spain,5296630.0


In [10]:
df = df[["date","country","group",'items']]
df.head()

,date,country,group,items
0,2021-01,Spain,Anxiolytics,4802610.0
1,2021-01,Spain,Antidepressants,3892410.0
2,2021-02,Spain,Anxiolytics,4619320.0
3,2021-02,Spain,Antidepressants,3720430.0
4,2021-03,Spain,Anxiolytics,5296630.0


In [11]:
# save to csv
df.to_csv("EDA_Spain_pharm.csv", index=False)